## ⚠️ IMPORTANT: Before You Run the Notebook

Complete the following setup steps before running any notebook in this lab.

**1. Create the Notebook Service**

If you have not already created a service by following the instructions on the Welcome page:

1. Go to **Connect → Create New Service**.
2. Under **Artifact Repository**, ensure `SNOWFLAKE.SNOWPARK.PYPI_SHARED_REPOSITORY` is selected
3. Expand **Service Settings**, Select Default compute pool: **SYSTEM_COMPUTE_POOL_CPU(CPU_X64_S)**
4. Set idle timeout: **2 hours**
5. For **external access integrations**, select `RETAIL_HOL_POSTGRES_EAI` (⚠️ Please **do not** skip this step)

Please do not change any other options. These options give the notebook permission to connect to your Postgres instance and install required python packages.

This is a one-time setup. It creates your personal notebook runtime, and all other notebooks in the workspace will connect to it automatically.

# Part 1: Openflow Gen2 Deployment

> **Run this notebook at the very start of the HoL session — before anything else.**
>
> Creating an Openflow deployment takes **~15 minutes** to reach ACTIVE state. Kick it off now so it runs in the background while you go through the introductions and Notebook 01. You'll return here to verify ACTIVE status before starting Notebook 02.

## What we'll create

| Object | Name | Purpose |
|--------|------|---------|
| Database/Schema | `OPENFLOW_DB.OPENFLOW_SCHEMA` | Hosts Openflow runtime and connector objects |
| Role | `OPENFLOW_ADMIN` | Manages deployments, runtimes, and connectors |
| Runtime Role | `POSTGRES_INGEST_SERVICE_ROLE` | Runtime execution identity with least-privilege access |
| Deployment | `SPCS_SOM` | Openflow deployment (Snowflake-managed SPCS compute) |
| Event Table | `OPENFLOW_DB.OPENFLOW_SCHEMA.EVENTS` | Runtime telemetry and logging |

## Prerequisites (pre-provisioned)

- Role: `RETAIL_HOL_ROLE`
- Warehouse: `RETAIL_HOL_WH`
- Databases: `RETAIL_HOL_ADMIN`, `RETAIL_HOL_ICEBERG`
- Network Policy: `RETAIL_HOL_NETWORK_POLICY`
- EAI: `RETAIL_HOL_POSTGRES_EAI`

## Step 1: Create the Openflow database and schema

Openflow objects (runtimes, connectors, event tables) live in a dedicated database and schema, separate from your data.

In [ ]:
%%sql -r dataframe_1
USE ROLE ACCOUNTADMIN;

CREATE DATABASE IF NOT EXISTS OPENFLOW_DB;
CREATE SCHEMA IF NOT EXISTS OPENFLOW_DB.OPENFLOW_SCHEMA;

## Step 2: Create the Openflow admin role

The `OPENFLOW_ADMIN` role manages Openflow infrastructure (deployments, runtimes, connectors). It is separate from the data access role (`RETAIL_HOL_ROLE`) to follow least-privilege principles.

In [ ]:
%%sql -r dataframe_2
CREATE ROLE IF NOT EXISTS OPENFLOW_ADMIN COMMENT = 'Manages Openflow Gen2 (SOM) infrastructure';
GRANT ROLE OPENFLOW_ADMIN TO USER "USER";

ALTER USER "USER" SET DEFAULT_SECONDARY_ROLES = ('ALL');

## Step 3: Grant privileges to OPENFLOW_ADMIN

The admin role needs account-level privileges (create deployment, create compute pool) and schema-level privileges (create runtime, create connector, create event table).

In [ ]:
%%sql -r dataframe_3
-- Account-level: allow creating deployments and compute pools
GRANT CREATE OPENFLOW DEPLOYMENT ON ACCOUNT TO ROLE OPENFLOW_ADMIN;
GRANT CREATE COMPUTE POOL ON ACCOUNT TO ROLE OPENFLOW_ADMIN;

-- Database-level: access to the Openflow database
GRANT USAGE ON DATABASE OPENFLOW_DB TO ROLE OPENFLOW_ADMIN;

-- Schema-level: create runtimes, connectors, and event tables
GRANT USAGE ON SCHEMA OPENFLOW_DB.OPENFLOW_SCHEMA TO ROLE OPENFLOW_ADMIN;
GRANT CREATE OPENFLOW RUNTIME ON SCHEMA OPENFLOW_DB.OPENFLOW_SCHEMA TO ROLE OPENFLOW_ADMIN;
GRANT CREATE OPENFLOW CONNECTOR ON SCHEMA OPENFLOW_DB.OPENFLOW_SCHEMA TO ROLE OPENFLOW_ADMIN;
GRANT CREATE EVENT TABLE ON SCHEMA OPENFLOW_DB.OPENFLOW_SCHEMA TO ROLE OPENFLOW_ADMIN;

## Step 4: Create and configure the runtime service role

The `POSTGRES_INGEST_SERVICE_ROLE` is the **execute-as role** for the runtime — it's the identity under which the connector operates. It needs:
- Read access to the source secret (Postgres password created in Notebook 01)
- Write access to the destination database (create schemas/tables)
- Access to the warehouse and EAI for connectivity

In [ ]:
%%sql -r dataframe_4
USE ROLE ACCOUNTADMIN;

-- Create the runtime service role
CREATE ROLE IF NOT EXISTS POSTGRES_INGEST_SERVICE_ROLE;

-- OPENFLOW_ADMIN needs this role to assign EXECUTE_AS_ROLE during runtime creation
GRANT ROLE POSTGRES_INGEST_SERVICE_ROLE TO ROLE OPENFLOW_ADMIN;

-- Access to the destination database (connector will create schemas and tables here)
GRANT USAGE ON DATABASE RETAIL_HOL_ICEBERG TO ROLE POSTGRES_INGEST_SERVICE_ROLE;
GRANT CREATE SCHEMA ON DATABASE RETAIL_HOL_ICEBERG TO ROLE POSTGRES_INGEST_SERVICE_ROLE;

-- Access to the secret's database and schema
GRANT USAGE ON DATABASE RETAIL_HOL_ADMIN TO ROLE POSTGRES_INGEST_SERVICE_ROLE;
GRANT USAGE ON SCHEMA RETAIL_HOL_ADMIN.SECRETS TO ROLE POSTGRES_INGEST_SERVICE_ROLE;

-- Warehouse access for processing
GRANT USAGE, OPERATE ON WAREHOUSE RETAIL_HOL_WH TO ROLE POSTGRES_INGEST_SERVICE_ROLE;

-- External Access Integration for network connectivity to Postgres
GRANT USAGE ON INTEGRATION RETAIL_HOL_POSTGRES_EAI TO ROLE POSTGRES_INGEST_SERVICE_ROLE;

## Step 5: Create the Openflow Deployment Gen2 and Event Table

A **deployment** is the top-level container for Openflow runtimes. For SPCS (Snowflake-managed), the deployment type is `SNOWFLAKE`. The **event table** captures runtime telemetry and logs for monitoring.

> **This step kicks off the ~15 minute deployment process.** Once you run the cells below ("Create the Deployment" and "Events Table"), proceed to `01-retail-hol-instance-creation-data-population.ipynb`.

In [ ]:
%%sql -r dataframe_5
USE ROLE OPENFLOW_ADMIN;

CREATE OPENFLOW DEPLOYMENT IF NOT EXISTS SPCS_SOM
  DEPLOYMENT_TYPE = 'SNOWFLAKE'
  DISPLAY_NAME = SPCS_SOM
  COMMENT = 'SPCS_SOM';

### Configure a Dedicated Openflow Event Table (Optional but recommended)

Openflow generates logs and metrics and sends them to a Snowflake event table. By default it uses the **account-level event table** (`SNOWFLAKE.TELEMETRY.EVENTS`), which is shared across all Snowflake features in your account.

You can optionally configure a **dedicated event table per deployment**. This is recommended because:

- **Query performance** — Openflow telemetry is isolated from account-wide noise, making queries faster and easier to write
- **Access control** — you can grant access to Openflow logs without exposing the full account event table
- **Easier monitoring** — all Openflow-specific logs, errors, and metrics are in one place
- **Observability dashboard** — the built-in Openflow Connector Observability dashboard works best with a dedicated event table, as it keeps Openflow telemetry separate from account-wide data for cleaner and faster queries

> If you skip this step, Openflow will fall back to `SNOWFLAKE.TELEMETRY.EVENTS` automatically. The cell below creates a dedicated event table and attaches it to the `SPCS_SOM` deployment.

In [ ]:
%%sql -r dataframe_10
// Create the Events table and attach it to the deployment
USE ROLE OPENFLOW_ADMIN;

CREATE EVENT TABLE IF NOT EXISTS OPENFLOW_DB.OPENFLOW_SCHEMA.EVENTS;

ALTER OPENFLOW DEPLOYMENT SPCS_SOM SET
  EVENT_TABLE = OPENFLOW_DB.OPENFLOW_SCHEMA.EVENTS;

## Recap: What We Built

In this notebook, we set up all the foundational Openflow infrastructure and kicked off the deployment that powers the CDC pipeline in Notebook 02.

### Infrastructure provisioned

| Object | Name | Purpose |
|--------|------|---------|
| Database & Schema | `OPENFLOW_DB.OPENFLOW_SCHEMA` | Dedicated namespace for all Openflow objects |
| Role | `OPENFLOW_ADMIN` | Manages deployments, runtimes, and connectors |
| Runtime Role | `POSTGRES_INGEST_SERVICE_ROLE` | Runtime execution identity with least-privilege access |
| Deployment | `SPCS_SOM` | Snowflake-managed SPCS compute container for Openflow |
| Event Table | `OPENFLOW_DB.OPENFLOW_SCHEMA.EVENTS` | Runtime telemetry and error logs |

### Key concepts covered

- **Openflow SOM** — managing Openflow infrastructure entirely through SQL (`CREATE/ALTER/DROP OPENFLOW DEPLOYMENT`), no NiFi canvas required
- **Least-privilege service role** — `POSTGRES_INGEST_SERVICE_ROLE` has only the permissions needed: read the secret, write to the destination database, use the EAI
- **Deployment lifecycle** — a deployment provisions Snowflake-managed SPCS compute; it takes ~15 minutes to become ACTIVE and is a prerequisite for creating runtimes
- **Event table** — attached to the deployment to capture telemetry, errors, and audit logs from all connectors running on this deployment

---

## Next Steps

Please proceed to notebook `01-retail-hol-instance-creation-data-population.ipynb`.